# TAAC-inspired TMS masking lab

Record or load a TMS-click WAV, isolate one pulse, synthesize white/click-derived/hybrid masking, schedule audible single pulses or rhythmic trains, and stream or export the result. This is an experimental auditory-control tool—not a stimulator controller or calibrated hearing-safety system. Start with low hardware volume and validate the complete headphone chain.

In [ ]:
from pathlib import Path
import sys
from dataclasses import asdict
import json, time
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import Audio, display, clear_output
NOTEBOOK_DIR = Path.cwd() if (Path.cwd()/'taac_audio.py').exists() else Path.cwd()/'in progress'/'TAAC reproduction'
sys.path.insert(0, str(NOTEBOOK_DIR))
from taac_audio import (Settings, RecordingSession, StreamingPlayer, audio_stats, crop_pulse,
    default_name, file_sha256, generate, list_input_devices, load_settings, read_wav,
    record, save_settings, suggest_pulse_bounds, write_wav)
ROOT = NOTEBOOK_DIR
for folder in ('generated', 'recordings', 'settings', 'assets'): (ROOT/folder).mkdir(exist_ok=True)
state = {'source': None, 'source_rate': None, 'pulse': None, 'pulse_rate': None, 'mixed': None, 'stats': None, 'source_path': None}
recorder, player = RecordingSession(), StreamingPlayer()
print('Working folder:', ROOT)

## 1. Record or load a click recording
Recording is optional. Use an appropriate microphone/interface and keep unsuitable equipment away from the coil. Stop capture manually or use timed recording.

In [ ]:
devices = list_input_devices()
device = W.Dropdown(options=[('System default', None)] + [(f'{i}: {n}', i) for i,n in devices], description='Input:', layout=W.Layout(width='700px'))
record_rate = W.Dropdown(options=[44100, 48000, 96000], value=96000, description='Rate:', layout=W.Layout(width='700px'))
record_duration = W.FloatSlider(min=.25, max=30, value=5, step=.25, description='Seconds:', layout=W.Layout(width='700px'))
upload = W.FileUpload(accept='.wav', multiple=False, description='Upload WAV', layout=W.Layout(width='340px'))
arm = W.Button(description='Start recording', button_style='danger', layout=W.Layout(width='220px'))
timed_record = W.Button(description='Record timed', button_style='warning', layout=W.Layout(width='220px'))
stop_record = W.Button(description='Stop & keep', layout=W.Layout(width='220px'))
discard_record = W.Button(description='Discard', layout=W.Layout(width='220px'))
record_status = W.Output()
def show_source(label):
    x, rate = state['source'], state['source_rate']; stats = audio_stats(x)
    with record_status:
        clear_output(); print(label, f'| {len(x)/rate:.3f}s at {rate} Hz | peak {stats["peak"]:.4f} | RMS {stats["rms"]:.4f} | clipped {stats["clipped"]}')
        display(Audio(x, rate=rate, normalize=False))
def on_upload(change):
    if not upload.value: return
    item = next(iter(upload.value.values())) if isinstance(upload.value, dict) else upload.value[0]
    path = ROOT/'recordings'/item['name']; path.write_bytes(item['content'])
    state['source'], state['source_rate'] = read_wav(path); state['source_path'] = path; show_source('Loaded '+item['name'])
def start_record(_):
    with record_status:
        clear_output(); recorder.start(record_rate.value, device.value); print('RECORDING — press Stop & keep')
def keep_recording(x):
    if not x.size:
        with record_status: print('No samples captured.')
        return
    path = ROOT/'recordings'/f'tms_click_{time.strftime("%Y%m%d_%H%M%S")}_{record_rate.value}Hz.wav'
    write_wav(path, x, record_rate.value, 24); state.update(source=x, source_rate=record_rate.value, source_path=path); show_source('Recorded '+path.name)
def timed_capture(_):
    with record_status: clear_output(); print(f'Recording for {record_duration.value:g} s…')
    keep_recording(record(record_duration.value, record_rate.value, device.value))
def finish_record(_):
    x = recorder.stop()
    if not x.size:
        with record_status: print('No samples captured.')
        return
    keep_recording(x)
def discard(_): recorder.stop(discard=True); state.update(source=None, source_rate=None, source_path=None);
upload.observe(on_upload, names='value'); arm.on_click(start_record); timed_record.on_click(timed_capture); stop_record.on_click(finish_record); discard_record.on_click(discard)
display(W.VBox([device, record_rate, record_duration, W.HBox([upload, arm, timed_record, stop_record, discard_record]), record_status]))

## 2. Select one pulse
Automatic boundaries surround the strongest transient. Adjust the long sliders after pressing **Suggest bounds**, then press **Apply crop**.

In [ ]:
start_ms = W.FloatSlider(min=0, max=1000, step=.05, description='Start ms:', continuous_update=False, layout=W.Layout(width='900px'))
end_ms = W.FloatSlider(min=.1, max=1000, value=32, step=.05, description='End ms:', continuous_update=False, layout=W.Layout(width='900px'))
suggest = W.Button(description='Suggest bounds'); apply_crop = W.Button(description='Apply crop', button_style='success')
pulse_output = W.Output()
def plot_selection():
    x, rate = state['source'], state['source_rate']; t = np.arange(x.size)/rate*1000
    with pulse_output:
        clear_output(); fig, ax = plt.subplots(figsize=(14,3)); ax.plot(t,x,lw=.7); ax.axvspan(start_ms.value,end_ms.value,color='tab:red',alpha=.25); ax.set(xlabel='Time (ms)',ylabel='Amplitude'); plt.show()
def suggest_bounds(_):
    if state['source'] is None: raise RuntimeError('Record or upload audio first')
    x, rate = state['source'], state['source_rate']; a,b=suggest_pulse_bounds(x,rate); duration=len(x)/rate*1000
    start_ms.max=end_ms.max=duration; start_ms.value=a/rate*1000; end_ms.value=b/rate*1000; plot_selection()
def crop(_):
    state['pulse']=crop_pulse(state['source'],start_ms.value,end_ms.value,state['source_rate']); state['pulse_rate']=state['source_rate']
    with pulse_output: print(f'Selected {len(state["pulse"])/state["pulse_rate"]*1000:.2f} ms pulse'); display(Audio(state['pulse'],rate=state['pulse_rate'],normalize=False))
suggest.on_click(suggest_bounds); apply_crop.on_click(crop)
display(W.VBox([start_ms,end_ms,W.HBox([suggest,apply_crop]),pulse_output]))

## 3. Generate masking and scheduled clicks
Probability applies to candidate single pulses or whole rhythmic trains. Pulses per train remains an independent rTMS-pattern parameter. All gains are digital amplitudes, not dB SPL.

In [ ]:
wide=W.Layout(width='900px'); half=W.Layout(width='440px')
duration=W.FloatSlider(min=.1,max=300,value=10,step=.1,description='Duration s:',layout=wide,continuous_update=False)
sample_rate=W.Dropdown(options=[44100,48000,96000],value=48000,description='Output Hz:',layout=half)
mode=W.ToggleButtons(options=['white','click-derived','hybrid'],value='hybrid',description='Noise:')
ratio=W.FloatSlider(min=0,max=1,value=.75,step=.01,description='Click ratio:',layout=wide)
broadband=W.Checkbox(value=True,description='Broadband click-derived noise'); layers=W.IntSlider(min=1,max=15,value=7,description='Layers:',layout=wide)
factors=W.FloatRangeSlider(min=.2,max=2.5,value=[.45,1.8],step=.01,description='Factors:',layout=wide)
seed=W.IntText(value=20260811,description='Seed:',layout=half); master=W.FloatSlider(min=0,max=1,value=.35,step=.005,description='Master:',layout=wide)
include=W.Checkbox(value=True,description='Include audible masking clicks'); click_gain=W.FloatSlider(min=0,max=50,value=5,step=.1,description='Click gain:',layout=wide)
schedule=W.ToggleButtons(options=[('Single pulse','single-pulse'),('Rhythmic','rhythmic')],description='Pattern:')
single_interval=W.FloatRangeSlider(min=.05,max=10,value=[.6,1.4],step=.01,description='Single IPI:',layout=wide)
rhythmic_hz=W.FloatSlider(min=.1,max=50,value=5,step=.1,description='Train Hz:',layout=wide); pulses=W.IntSlider(min=1,max=100,value=5,description='Pulses/train:',layout=wide)
train_interval=W.FloatSlider(min=0,max=60,value=2,step=.05,description='Inter-train s:',layout=wide); probability=W.FloatSlider(min=0,max=1,value=.5,step=.01,description='Probability:',layout=wide)
jitter=W.FloatSlider(min=0,max=.1,value=0,step=.001,description='Jitter s:',layout=wide); amp_var=W.FloatSlider(min=0,max=.9,value=.15,step=.01,description='Amp variation:',layout=wide)
generate_button=W.Button(description='Generate / validate',button_style='success',layout=W.Layout(width='300px')); generation_output=W.Output()
def current_settings():
    return Settings(sample_rate=sample_rate.value,duration_s=duration.value,seed=seed.value,noise_mode=mode.value,click_ratio=ratio.value,broadband=broadband.value,layers=layers.value,min_factor=factors.value[0],max_factor=factors.value[1],master_gain=master.value,include_clicks=include.value,click_gain=click_gain.value,schedule=schedule.value,single_min_interval_s=single_interval.value[0],single_max_interval_s=single_interval.value[1],rhythmic_hz=rhythmic_hz.value,pulses_per_train=pulses.value,inter_train_interval_s=train_interval.value,probability=probability.value,jitter_s=jitter.value,amplitude_variation=amp_var.value)
def do_generate(_=None):
    if state['pulse'] is None: raise RuntimeError('Select and apply a pulse first')
    s=current_settings(); state['mixed'],state['stats']=generate(state['pulse'],state['pulse_rate'],s)
    with generation_output:
        clear_output(); print(json.dumps({k:v for k,v in state['stats'].items() if k!='settings'},indent=2)); display(Audio(state['mixed'],rate=s.sample_rate,normalize=False))
generate_button.on_click(do_generate)
display(W.VBox([duration,sample_rate,mode,ratio,broadband,layers,factors,seed,master,include,click_gain,schedule,single_interval,rhythmic_hz,pulses,train_interval,probability,jitter,amp_var,generate_button,generation_output]))

## 4. Continuous transport and export
Infinite mode streams the validated buffer continuously. Pause immediately outputs silence while retaining position; Resume continues from that position.

In [ ]:
play=W.Button(description='Play',button_style='success'); pause=W.Button(description='Pause'); resume=W.Button(description='Resume'); stop=W.Button(description='Stop',button_style='danger')
loop=W.Checkbox(value=True,description='Infinite'); save_wav_button=W.Button(description='Export WAV'); save_json_button=W.Button(description='Save settings')
settings_upload=W.FileUpload(accept='.json',multiple=False,description='Upload settings'); transport_output=W.Output()
def ensure_audio():
    if state['mixed'] is None: do_generate()
def on_play(_): ensure_audio(); player.play(state['mixed'],current_settings().sample_rate,loop.value);
def on_pause(_): player.pause()
def on_resume(_): player.resume()
def on_stop(_): player.stop()
def export_wav(_):
    ensure_audio(); s=current_settings(); path=write_wav(ROOT/'generated'/(default_name(s)+'.wav'),state['mixed'],s.sample_rate,24)
    with transport_output: clear_output(); print('Saved',path)
def export_json(_):
    s=current_settings(); extra={'source':Path(state['source_path']).name if state['source_path'] else None,'source_sha256':file_sha256(state['source_path']) if state['source_path'] else None,'pulse_bounds_ms':[start_ms.value,end_ms.value],'statistics':state['stats']}
    path=save_settings(ROOT/'settings'/(default_name(s)+'.json'),s,extra)
    with transport_output: clear_output(); print('Saved',path)
def import_json(change):
    if not settings_upload.value:return
    item=next(iter(settings_upload.value.values())) if isinstance(settings_upload.value,dict) else settings_upload.value[0]; path=ROOT/'settings'/item['name']; path.write_bytes(item['content']); s,extra=load_settings(path)
    duration.value=s.duration_s; sample_rate.value=s.sample_rate; seed.value=s.seed; mode.value=s.noise_mode; ratio.value=s.click_ratio; broadband.value=s.broadband; layers.value=s.layers; factors.value=[s.min_factor,s.max_factor]; master.value=s.master_gain; include.value=s.include_clicks; click_gain.value=s.click_gain; schedule.value=s.schedule; single_interval.value=[s.single_min_interval_s,s.single_max_interval_s]; rhythmic_hz.value=s.rhythmic_hz; pulses.value=s.pulses_per_train; train_interval.value=s.inter_train_interval_s; probability.value=s.probability; jitter.value=s.jitter_s; amp_var.value=s.amplitude_variation
    with transport_output: clear_output(); print('Loaded',path,'— regenerate before playback')
play.on_click(on_play);pause.on_click(on_pause);resume.on_click(on_resume);stop.on_click(on_stop);save_wav_button.on_click(export_wav);save_json_button.on_click(export_json);settings_upload.observe(import_json,names='value')
display(W.VBox([W.HBox([play,pause,resume,stop,loop]),W.HBox([save_wav_button,save_json_button,settings_upload]),transport_output]))

## Session notes

- Retune and revalidate masking after changes to stimulation intensity, coil/target, foam, microphone position, headphones, or room.
- Auditory masking does not control scalp sensation, vibration, expectancy, or muscle artifacts. Plan a separate somatosensory/sham control when interpretation requires it.
- Keep the original click recording. `generated/`, `recordings/`, and `settings/` are intentionally ignored and can be cleaned when no longer needed.